[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter8/image-retrieval-siglip.ipynb)

# Chapter 8: Image Retrieval with SigLIP

This notebook demonstrates image-based retrieval using Google's SigLIP model for multimodal RAG applications.

**What you'll learn:**
- Load and use Google's SigLIP model for joint image-text embeddings
- Encode images and text queries into a shared embedding space
- Retrieve the most relevant images for a text query using sigmoid scoring

**Prerequisites:** `pip install torch transformers Pillow requests`

In [1]:
import torch
import torch.nn.functional as F
from PIL import Image
import requests
from io import BytesIO
from transformers import AutoModel, AutoProcessor

## Step 1: load the SigLip model from Huggingface and 

We load Google's SigLIP model, which maps both images and text into a shared embedding space so we can find images by describing them in words.

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/siglip-so400m-patch14-384"
model = AutoModel.from_pretrained(model_id).to(device)
processor = AutoProcessor.from_pretrained(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Step 2: load images and encode them with SigLip

We download a diverse set of images and encode each one into SigLIP's embedding space, building the "database" we'll search against.

In [3]:
import time

def load_image(url, max_retries=3, retry_delay=2):
    """Load image from URL with retry logic."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert("RGB")
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Attempt {attempt + 1} failed for {url.split('/')[-1]}: {e}. Retrying...")
                time.sleep(retry_delay * (attempt + 1))
            else:
                print(f"Failed to load {url.split('/')[-1]} after {max_retries} attempts: {e}")
                return None

# Using reliable image sources (Unsplash) to avoid Wikimedia rate limiting
image_urls = [
    "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=800",  # Cat
    "https://images.unsplash.com/photo-1574158622682-e40e69881006?w=800",  # Another cat
    "https://images.unsplash.com/photo-1574071318508-1cdbab80d002?w=800",  # Margherita pizza (classic Italian)
    "https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=800",  # Mountain
    "https://images.unsplash.com/photo-1449034446853-66c86144b0ad?w=800",  # Golden Gate Bridge
    "https://images.unsplash.com/photo-1579952363873-27f3bade9f55?w=800",  # Soccer ball
    "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=800",  # Dog
]

# Load images with small delay between requests
images = []
for i, url in enumerate(image_urls):
    if i > 0:
        time.sleep(0.3)
    img = load_image(url)
    images.append(img)

# Filter out any that failed
valid_pairs = [(url, img) for url, img in zip(image_urls, images) if img is not None]
if len(valid_pairs) < len(image_urls):
    print(f"Warning: {len(image_urls) - len(valid_pairs)} images failed to load")
    image_urls = [url for url, _ in valid_pairs]
    images = [img for _, img in valid_pairs]

if not images:
    raise ValueError("No images could be loaded. Check your internet connection.")

print(f"Loaded {len(images)} images")

image_inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    image_features = model.get_image_features(**image_inputs)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

Loaded 7 images


## Step 3: run queries

We encode text queries into the same embedding space and use sigmoid scoring to retrieve the images that best match each description.

In [4]:
def retrieve_images(query_text, top_k=1):
    """
    Retrieves images matching the query using SigLIP.
    Returns both absolute probabilities (independent) and relative probabilities (softmax).
    """
    print(f"\nQuerying for: '{query_text}'")
    
    # Preprocess text
    # SigLIP prefers explicit 'max_length' padding for consistent tensor shapes
    text_inputs = processor(text=[query_text], return_tensors="pt", padding="max_length").to(device)
    
    with torch.no_grad():
        # 1. Get text embeddings & Normalize
        text_features = model.get_text_features(**text_inputs)
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        # 2. Calculate Dot Product (Raw Similarity)
        # Shape: [1, num_images]
        raw_scores = (text_features @ image_features.t())

        # 3. Apply Model Scaling & Bias
        # SigLIP trains these parameters to align the vector space
        logit_scale = model.logit_scale.exp()
        logit_bias = model.logit_bias
        logits = (raw_scores * logit_scale) + logit_bias

        # 4. Calculate Probabilities using Sigmoid
        sigmoid_probs = torch.sigmoid(logits)

    # We rank by the Softmax
    top_probs, indices = torch.topk(sigmoid_probs, top_k)    
    results = []
    for i, idx in enumerate(indices[0]):
        index_val = idx.item()        
        results.append({
            "url": image_urls[index_val],
            "score": top_probs[0][i].item(),
            "image_index": index_val
        })
    
    return results

In [5]:
# Search 1
results_cat = retrieve_images("A cute kitten", top_k=3)
for res in results_cat:
    print(f"Match (Probability: {res['score']:.4f}): {res['url']}")


Querying for: 'A cute kitten'
Match (Probability: 0.0476): https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=800
Match (Probability: 0.0196): https://images.unsplash.com/photo-1574158622682-e40e69881006?w=800
Match (Probability: 0.0000): https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=800


In [6]:
# Search 2
results_pizza = retrieve_images("A round Margharitta Pizza", top_k=1)
for res in results_pizza:
    print(f"Match (Probability: {res['score']:.4f}): {res['url']}")


Querying for: 'A round Margharitta Pizza'
Match (Probability: 0.8323): https://images.unsplash.com/photo-1574071318508-1cdbab80d002?w=800


In [7]:
# Search 3
results_pizza = retrieve_images("Delicious italian food with cheese", top_k=1)
for res in results_pizza:
    print(f"Match (Probability: {res['score']:.4f}): {res['url']}")


Querying for: 'Delicious italian food with cheese'
Match (Probability: 0.0135): https://images.unsplash.com/photo-1574071318508-1cdbab80d002?w=800
